# Left ventricular segmentation with U-Net
Train and evaluate a binary LV endocardium segmentation model on CAMUS. Patients, rather than individual frames, are split between training, validation and test sets.

In [ ]:
from pathlib import Path
import os, random, json
import numpy as np
import pandas as pd
import cv2
import SimpleITK as sitk
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
SIZE, BATCH, EPOCHS = 256, 8, 40
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists(): ROOT = ROOT.parent
TRAINING = Path(os.environ.get("CAMUS_TRAINING_DIR", str(ROOT / "data/camus/training")))
OUTPUT = ROOT / "results"
OUTPUT.mkdir(exist_ok=True)
assert TRAINING.is_dir(), f"CAMUS training directory not found: {TRAINING}"

## Load annotated images and split by patient

In [ ]:
def read_image(path):
    arr = sitk.GetArrayFromImage(sitk.ReadImage(str(path)))
    return arr[0] if arr.ndim == 3 else arr

def process_image(path):
    a = cv2.resize(read_image(path).astype("float32"), (SIZE, SIZE))
    return (a - a.min()) / (a.max() - a.min() + 1e-8)

def process_mask(path):
    a = cv2.resize(read_image(path).astype("uint8"), (SIZE, SIZE), interpolation=cv2.INTER_NEAREST)
    return (a == 1).astype("float32")  # CAMUS LV endocardium label

records = []
for patient_dir in sorted(TRAINING.glob("patient*")):
    for mask_path in sorted(patient_dir.glob("*_gt.mhd")):
        image_path = mask_path.with_name(mask_path.name.replace("_gt.mhd", ".mhd"))
        if image_path.exists():
            records.append({"patient": patient_dir.name, "image": image_path, "mask": mask_path})
df = pd.DataFrame(records)
assert len(df) >= 10, "No matching CAMUS .mhd image/mask pairs found"
first, rest = next(GroupShuffleSplit(n_splits=1, test_size=.30, random_state=SEED).split(df, groups=df.patient))
remaining = df.iloc[rest]
val_i, test_i = next(GroupShuffleSplit(n_splits=1, test_size=.50, random_state=SEED).split(remaining, groups=remaining.patient))
train_df, val_df, test_df = df.iloc[first], remaining.iloc[val_i], remaining.iloc[test_i]
assert len(set(train_df.patient) & set(val_df.patient)) == 0
assert len(set(train_df.patient) & set(test_df.patient)) == 0
assert len(set(val_df.patient) & set(test_df.patient)) == 0
print([(name, len(split), split.patient.nunique()) for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]])

def load(split):
    x = np.stack([process_image(p) for p in split.image]).astype("float32")[..., None]
    y = np.stack([process_mask(p) for p in split["mask"]]).astype("float32")[..., None]
    return x, y

x_train, y_train = load(train_df)
x_val, y_val = load(val_df)
x_test, y_test = load(test_df)

## U-Net model

In [ ]:
def block(x, filters):
    for _ in range(2):
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
    return x

def build_unet():
    inputs = keras.Input((SIZE, SIZE, 1))
    x, skips = inputs, []
    for filters in [16, 32, 64, 128]:
        x = block(x, filters)
        skips.append(x)
        x = layers.MaxPooling2D()(x)
    x = block(x, 256)
    for filters, skip in zip([128, 64, 32, 16], reversed(skips)):
        x = layers.Conv2DTranspose(filters, 2, strides=2, padding="same")(x)
        x = block(layers.Concatenate()([x, skip]), filters)
    return keras.Model(inputs, layers.Conv2D(1, 1, activation="sigmoid")(x))

def dice(y_true, y_pred):
    intersection = tf.reduce_sum(y_true * y_pred, axis=(1,2,3))
    total = tf.reduce_sum(y_true + y_pred, axis=(1,2,3))
    return tf.reduce_mean((2 * intersection + 1) / (total + 1))

def loss(y_true, y_pred):
    return tf.reduce_mean(keras.losses.binary_crossentropy(y_true, y_pred)) + 1 - dice(y_true, y_pred)

model = build_unet()
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=loss, metrics=[dice])
model.summary()

## Training

In [ ]:
checkpoint = OUTPUT / "best_unet.weights.h5"
callbacks = [
    keras.callbacks.ModelCheckpoint(str(checkpoint), monitor="val_dice", mode="max", save_best_only=True, save_weights_only=True),
    keras.callbacks.EarlyStopping(monitor="val_dice", mode="max", patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=4, factor=.5)
]
history = model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=EPOCHS, batch_size=BATCH, callbacks=callbacks)
model.load_weights(str(checkpoint))
pd.DataFrame(history.history).to_csv(OUTPUT / "history.csv", index=False)

## Evaluation on held-out patients

In [ ]:
probabilities = model.predict(x_test, batch_size=BATCH)
predictions = probabilities >= .5
truth = y_test >= .5
rows = []
for prediction, target in zip(predictions, truth):
    p, t = prediction[..., 0], target[..., 0]
    overlap = np.logical_and(p, t).sum()
    union = np.logical_or(p, t).sum()
    rows.append({"dice": (2 * overlap + 1e-8)/(p.sum() + t.sum() + 1e-8),
                 "iou": (overlap + 1e-8)/(union + 1e-8)})
scores = pd.DataFrame(rows)
print(scores.agg(["mean", "std"]).round(4))
scores.to_csv(OUTPUT / "test_metrics.csv", index=False)
fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for i in range(min(3, len(x_test))):
    axes[i,0].imshow(x_test[i,:,:,0], cmap="gray")
    axes[i,1].imshow(truth[i,:,:,0], cmap="gray")
    axes[i,2].imshow(predictions[i,:,:,0], cmap="gray")
for ax in axes.flat: ax.axis("off")
for ax, title in zip(axes[0], ["Ultrasound", "Ground truth", "Prediction"]): ax.set_title(title)
plt.tight_layout()
plt.savefig(OUTPUT / "predictions.png", dpi=140)
plt.show()